# FIT5202 2026 S2 Assignment 1 : Analysing Parking & Congestion Patterns in Melbourne

## Table of Contents
* [Part 1 : Data Loading and Transformation](#part-1)  
    - [1.1 Data Preparation and Loading](#1.1)  
    - [1.2 Data Cleansing](#1.2)
    - [1.3 Partitition and Spark UI Evidence](#1.3)    
* [Part 2 : Working with DataFrames](#2-dataframes)  
    - [2.1-2.5 Query/Analysis](#2-dataframes)  
    - [2.6 Exploratory Analysis](#2.6)  
* [Part 3 : Query Optimisation](#part-3)  

Note: Feel free to add Code/Markdown cells as you need.

# Part 1 : Data Loading and Transformation (10%) <a class="anchor" name="part-1"></a>
In this section, you need to create data frames from the given datasets, perform partitioning and use various operations to answer the queries.  

## 1.1 Data Preparation and Loading <a class="anchor" name="1.1"></a>
1. Configure the Spark session to run locally using four cores;
1. Assign an appropriate application name to the Spark session;
1. Set the session timezone explicitly to 'Australia/Melbourne';
1. Define and apply an explicit StructType schema when loading each CSV and JSON dataset. 
1. Print the schema and row count for each DataFrame;
1. Display the first five rows of each DataFrame to verify successful data loading.



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType
)

student_auth_code = "kzha0139"

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName(f"FIT5202-A1-{student_auth_code}")
    .config("spark.sql.session.timeZone", "Australia/Melbourne")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", sc.master)
print("Application name:", sc.appName)
print(
    "Session timezone:",
    spark.conf.get("spark.sql.session.timeZone")
)

Spark version: 4.1.1
Spark master: local[4]
Application name: FIT5202-A1-kzha0139
Session timezone: Australia/Melbourne


In [2]:
import os

DATA_DIR = (
    "fit5202_2026_s2_dataset_v2/"
    "fit5202_2026_s2_dataset_v2"
)

PARKING_PATH = f"{DATA_DIR}/sensordata.csv"
TRAFFIC_PATH = f"{DATA_DIR}/traffic_count.csv"
AREA_PATH = f"{DATA_DIR}/area.json"
STREET_PATH = f"{DATA_DIR}/street.json"

data_paths = {
    "Parking": PARKING_PATH,
    "Traffic": TRAFFIC_PATH,
    "Area": AREA_PATH,
    "Street": STREET_PATH
}

print("Notebook working directory:", os.getcwd())

for name, path in data_paths.items():
    print(f"{name}: exists={os.path.exists(path)}, path={path}")

Notebook working directory: /home/student
Parking: exists=True, path=fit5202_2026_s2_dataset_v2/fit5202_2026_s2_dataset_v2/sensordata.csv
Traffic: exists=True, path=fit5202_2026_s2_dataset_v2/fit5202_2026_s2_dataset_v2/traffic_count.csv
Area: exists=True, path=fit5202_2026_s2_dataset_v2/fit5202_2026_s2_dataset_v2/area.json
Street: exists=True, path=fit5202_2026_s2_dataset_v2/fit5202_2026_s2_dataset_v2/street.json


In [3]:
parking_schema = StructType([
    StructField("deviceid", StringType(), True),
    StructField("arrivaltime", StringType(), True),
    StructField("departuretime", StringType(), True),
    StructField("streetmarker", StringType(), True),
    StructField("signplateid", StringType(), True),
    StructField("sign", StringType(), True),
    StructField("streetid", IntegerType(), True),
    StructField("betweenstreet1id", IntegerType(), True),
    StructField("betweenstreet2id", IntegerType(), True),
    StructField("sideofstreet", IntegerType(), True),
    StructField("sidename", StringType(), True),
    StructField("bayid", IntegerType(), True),
    StructField("inviolation", BooleanType(), True),
    StructField("vehiclepresent", BooleanType(), True),
    StructField("areaid", IntegerType(), True)
])

In [4]:
traffic_schema = StructType([
    StructField("d_year", IntegerType(), True),
    StructField("d_month", IntegerType(), True),
    StructField("d_day", IntegerType(), True),
    StructField("d_hour", IntegerType(), True),
    StructField("d_minute", IntegerType(), True),
    StructField("areaid", IntegerType(), True),
    StructField("traffic_count", IntegerType(), True)
])

In [5]:
area_record_schema = StructType([
    StructField("areaid", IntegerType(), True),
    StructField("areaname", StringType(), True)
])

area_file_schema = StructType([
    StructField(
        "area",
        ArrayType(area_record_schema),
        True
    )
])

In [6]:
street_record_schema = StructType([
    StructField("streetid", IntegerType(), True),
    StructField("streetname", StringType(), True)
])

street_file_schema = StructType([
    StructField(
        "street",
        ArrayType(street_record_schema),
        True
    )
])

In [7]:
parking_raw_df = (
    spark.read
    .option("header", True)
    .schema(parking_schema)
    .csv(PARKING_PATH)
)

traffic_raw_df = (
    spark.read
    .option("header", True)
    .schema(traffic_schema)
    .csv(TRAFFIC_PATH)
)

In [8]:
area_nested_df = (
    spark.read
    .option("multiLine", True)
    .schema(area_file_schema)
    .json(AREA_PATH)
)

area_raw_df = (
    area_nested_df
    .select(F.explode("area").alias("area_record"))
    .select("area_record.*")
)

street_nested_df = (
    spark.read
    .option("multiLine", True)
    .schema(street_file_schema)
    .json(STREET_PATH)
)

street_raw_df = (
    street_nested_df
    .select(F.explode("street").alias("street_record"))
    .select("street_record.*")
)

In [9]:
dataframes = {
    "Parking DataFrame": parking_raw_df,
    "Traffic DataFrame": traffic_raw_df,
    "Area DataFrame": area_raw_df,
    "Street DataFrame": street_raw_df
}

for name, dataframe in dataframes.items():
    print("=" * 80)
    print(name)
    print("=" * 80)

    dataframe.printSchema()
    dataframe.show(5, truncate=False)

Parking DataFrame
root
 |-- deviceid: string (nullable = true)
 |-- arrivaltime: string (nullable = true)
 |-- departuretime: string (nullable = true)
 |-- streetmarker: string (nullable = true)
 |-- signplateid: string (nullable = true)
 |-- sign: string (nullable = true)
 |-- streetid: integer (nullable = true)
 |-- betweenstreet1id: integer (nullable = true)
 |-- betweenstreet2id: integer (nullable = true)
 |-- sideofstreet: integer (nullable = true)
 |-- sidename: string (nullable = true)
 |-- bayid: integer (nullable = true)
 |-- inviolation: boolean (nullable = true)
 |-- vehiclepresent: boolean (nullable = true)
 |-- areaid: integer (nullable = true)

+--------+----------------------+----------------------+------------+-----------+----+--------+----------------+----------------+------------+--------+-----+-----------+--------------+------+
|deviceid|arrivaltime           |departuretime         |streetmarker|signplateid|sign|streetid|betweenstreet1id|betweenstreet2id|sideofstreet

In [10]:
row_counts = {}

for name, dataframe in dataframes.items():
    print(f"Counting {name}...")

    row_count = dataframe.count()
    row_counts[name] = row_count

    print(f"{name} row count: {row_count:,}")

Counting Parking DataFrame...
Parking DataFrame row count: 72,912,582
Counting Traffic DataFrame...
Traffic DataFrame row count: 10,956,114
Counting Area DataFrame...
Area DataFrame row count: 38
Counting Street DataFrame...
Street DataFrame row count: 126


In [11]:
for name, row_count in row_counts.items():
    print(f"{name}: {row_count:,} rows")

Parking DataFrame: 72,912,582 rows
Traffic DataFrame: 10,956,114 rows
Area DataFrame: 38 rows
Street DataFrame: 126 rows


### 1.1 Loading Summary

A Spark session was configured to run locally using four cores with the
application name `FIT5202-A1-kzha0139`. The session timezone was explicitly
set to `Australia/Melbourne`.

Explicit `StructType` schemas were defined and applied when loading all four
CSV and JSON datasets. The area and street JSON arrays were expanded into
row-based lookup DataFrames. The displayed schemas, row counts and sample
records confirm that all datasets were loaded successfully.

### 1.2 Data Cleansing and Validation <a class="anchor" name="1.2"></a>
The following tasks must be completed to ensure data quality and consistency (feel free to add code block as you see fit):
1) Identify the missing-value representations within the dataset and convert them to null. Briefly state which values were treated as missing.


In [12]:
from pyspark.sql.types import StringType

candidate_missing_tokens = [
    "na",
    "n/a",
    "null",
    "none",
    "unknown",
    "-",
    "?"
]


def profile_missing_values(dataframe, dataframe_name):
    """
    Print non-zero counts of:
    1. existing Spark null values;
    2. blank or whitespace-only strings;
    3. common text-based missing-value tokens.

    The aggregation is completed in one pass through each DataFrame.
    """

    aggregation_expressions = []

    for field in dataframe.schema.fields:
        column_name = field.name

        aggregation_expressions.append(
            F.sum(
                F.when(
                    F.col(column_name).isNull(),
                    1
                ).otherwise(0)
            ).alias(f"{column_name}__null")
        )

        if isinstance(field.dataType, StringType):
            normalised_value = F.lower(F.trim(F.col(column_name)))

            aggregation_expressions.append(
                F.sum(
                    F.when(
                        normalised_value == "",
                        1
                    ).otherwise(0)
                ).alias(f"{column_name}__blank")
            )

            aggregation_expressions.append(
                F.sum(
                    F.when(
                        normalised_value.isin(
                            candidate_missing_tokens
                        ),
                        1
                    ).otherwise(0)
                ).alias(f"{column_name}__token")
            )

    result = (
        dataframe
        .agg(*aggregation_expressions)
        .first()
        .asDict()
    )

    print("=" * 70)
    print(dataframe_name)
    print("=" * 70)

    non_zero_result_found = False

    for metric_name, metric_count in result.items():
        if metric_count and metric_count > 0:
            print(f"{metric_name}: {metric_count:,}")
            non_zero_result_found = True

    if not non_zero_result_found:
        print("No null, blank or candidate missing values found.")

In [13]:
profile_missing_values(
    parking_raw_df,
    "Parking DataFrame"
)

profile_missing_values(
    traffic_raw_df,
    "Traffic DataFrame"
)

profile_missing_values(
    area_raw_df,
    "Area DataFrame"
)

profile_missing_values(
    street_raw_df,
    "Street DataFrame"
)

Parking DataFrame
signplateid__null: 25,195,820
sign__null: 25,195,820
betweenstreet2id__null: 10,606,993
Traffic DataFrame
No null, blank or candidate missing values found.
Area DataFrame
areaname__blank: 1
Street DataFrame
No null, blank or candidate missing values found.


In [14]:
area_raw_df.filter(
    F.trim(F.col("areaname")) == ""
).show(truncate=False)

+------+--------+
|areaid|areaname|
+------+--------+
|7     |        |
+------+--------+



In [15]:
def convert_blank_strings_to_null(dataframe):
    """
    Convert empty or whitespace-only strings to Spark null while
    preserving all non-empty values and existing null values.
    """

    cleaned_dataframe = dataframe

    for field in dataframe.schema.fields:
        if isinstance(field.dataType, StringType):
            column_name = field.name

            cleaned_dataframe = cleaned_dataframe.withColumn(
                column_name,
                F.when(
                    F.trim(F.col(column_name)) == "",
                    F.lit(None).cast(StringType())
                ).otherwise(F.col(column_name))
            )

    return cleaned_dataframe

In [16]:
parking_clean_df = convert_blank_strings_to_null(
    parking_raw_df
)

traffic_clean_df = convert_blank_strings_to_null(
    traffic_raw_df
)

area_clean_df = convert_blank_strings_to_null(
    area_raw_df
)

street_clean_df = convert_blank_strings_to_null(
    street_raw_df
)

In [17]:
print(
    "Raw blank AreaName count:",
    area_raw_df.filter(
        F.trim(F.col("areaname")) == ""
    ).count()
)

print(
    "Clean null AreaName count:",
    area_clean_df.filter(
        F.col("areaname").isNull()
    ).count()
)

area_clean_df.filter(
    F.col("areaname").isNull()
).show(truncate=False)

Raw blank AreaName count: 1
Clean null AreaName count: 1
+------+--------+
|areaid|areaname|
+------+--------+
|7     |NULL    |
+------+--------+



The missing-value profile identified empty CSV fields and one blank string as
the missing-value representations in the supplied datasets. Empty CSV fields
in `signplateid`, `sign` and `betweenstreet2id` were already interpreted as
Spark null values during loading. One blank `areaname` value was identified
and converted to null.

No occurrences of `NA`, `N/A`, `null`, `none`, `unknown`, `-` or `?` were
found. Existing null values were retained, while empty and whitespace-only
strings were converted to Spark null values. The original DataFrames were
kept unchanged, and separate cleaned DataFrames were created for subsequent
tasks.

2) Parse the 'ArrivalTime' and 'DepartureTime' columns and convert them to timestamp format. 

In [18]:
timestamp_pattern = "MM/dd/yyyy hh:mm:ss a"

parking_clean_df = (
    parking_clean_df
    .withColumn(
        "arrival_time_raw",
        F.col("arrivaltime")
    )
    .withColumn(
        "departure_time_raw",
        F.col("departuretime")
    )
    .withColumn(
        "arrivaltime",
        F.try_to_timestamp(
            F.col("arrivaltime"),
            F.lit(timestamp_pattern)
        )
    )
    .withColumn(
        "departuretime",
        F.try_to_timestamp(
            F.col("departuretime"),
            F.lit(timestamp_pattern)
        )
    )
)

In [19]:
parking_clean_df.select(
    "arrival_time_raw",
    "arrivaltime",
    "departure_time_raw",
    "departuretime"
).show(5, truncate=False)

+----------------------+-------------------+----------------------+-------------------+
|arrival_time_raw      |arrivaltime        |departure_time_raw    |departuretime      |
+----------------------+-------------------+----------------------+-------------------+
|10/23/2018 06:49:29 AM|2018-10-23 06:49:29|10/23/2018 06:54:24 AM|2018-10-23 06:54:24|
|02/07/2018 07:30:00 PM|2018-02-07 19:30:00|02/07/2018 07:56:23 PM|2018-02-07 19:56:23|
|11/22/2018 07:22:38 AM|2018-11-22 07:22:38|11/22/2018 07:28:22 AM|2018-11-22 07:28:22|
|09/18/2018 09:24:56 PM|2018-09-18 21:24:56|09/18/2018 09:27:44 PM|2018-09-18 21:27:44|
|01/04/2018 12:00:00 AM|2018-01-04 00:00:00|01/04/2018 03:45:46 AM|2018-01-04 03:45:46|
+----------------------+-------------------+----------------------+-------------------+
only showing top 5 rows


In [20]:
parking_clean_df.select(
    "arrival_time_raw",
    "arrivaltime",
    "departure_time_raw",
    "departuretime"
).printSchema()

root
 |-- arrival_time_raw: string (nullable = true)
 |-- arrivaltime: timestamp (nullable = true)
 |-- departure_time_raw: string (nullable = true)
 |-- departuretime: timestamp (nullable = true)



3) Convert the “DeviceId” to an appropriate numeric type. 

In [21]:
parking_clean_df = (
    parking_clean_df
    .withColumn(
        "device_id_raw",
        F.col("deviceid")
    )
    .withColumn(
        "deviceid",
        F.expr("try_cast(deviceid AS BIGINT)")
    )
)

In [22]:
parking_clean_df.select(
    "device_id_raw",
    "deviceid"
).show(5, truncate=False)

+-------------+--------+
|device_id_raw|deviceid|
+-------------+--------+
|17193        |17193   |
|17193        |17193   |
|17195        |17195   |
|17195        |17195   |
|17195        |17195   |
+-------------+--------+
only showing top 5 rows


In [23]:
parking_clean_df.select(
    "device_id_raw",
    "deviceid"
).printSchema()

root
 |-- device_id_raw: string (nullable = true)
 |-- deviceid: long (nullable = true)



4) Calculate a new “DurationMinutes” column using the arrival and departure timestamp. Store the result as a DoubleType value and rounded to two decimal places.

In [24]:
parking_clean_df = parking_clean_df.withColumn(
    "DurationMinutes",
    F.round(
        (
            F.col("departuretime").cast("long")
            - F.col("arrivaltime").cast("long")
        ) / F.lit(60.0),
        2
    ).cast("double")
)

In [25]:
parking_clean_df.select(
    "arrivaltime",
    "departuretime",
    "DurationMinutes"
).show(10, truncate=False)

+-------------------+-------------------+---------------+
|arrivaltime        |departuretime      |DurationMinutes|
+-------------------+-------------------+---------------+
|2018-10-23 06:49:29|2018-10-23 06:54:24|4.92           |
|2018-02-07 19:30:00|2018-02-07 19:56:23|26.38          |
|2018-11-22 07:22:38|2018-11-22 07:28:22|5.73           |
|2018-09-18 21:24:56|2018-09-18 21:27:44|2.8            |
|2018-01-04 00:00:00|2018-01-04 03:45:46|225.77         |
|2018-07-13 00:00:00|2018-07-13 06:06:43|366.72         |
|2018-01-29 19:07:39|2018-01-29 19:11:53|4.23           |
|2018-10-20 06:29:15|2018-10-20 07:15:02|45.78          |
|2018-08-06 18:42:28|2018-08-06 18:49:11|6.72           |
|2018-01-03 07:02:17|2018-01-03 07:05:08|2.85           |
+-------------------+-------------------+---------------+
only showing top 10 rows


In [26]:
parking_clean_df.select(
    "DurationMinutes"
).printSchema()

root
 |-- DurationMinutes: double (nullable = true)



5) Identify and report records where the departure time is earlier than the arrival time. Briefly discuss any pattern observed and state how these records will be handled in duration-based queries.

In [27]:
parking_clean_df = parking_clean_df.withColumn(
    "DepartureBeforeArrival",
    F.when(
        F.col("arrivaltime").isNotNull()
        & F.col("departuretime").isNotNull(),
        F.col("departuretime") < F.col("arrivaltime")
    ).otherwise(False)
)

In [28]:
from pyspark import StorageLevel

negative_duration_df = (
    parking_clean_df
    .filter(F.col("DepartureBeforeArrival"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

In [29]:
negative_duration_summary_df = negative_duration_df.agg(
    F.count("*").alias("NegativeDurationRecords"),
    F.round(
        F.min("DurationMinutes"),
        2
    ).alias("MostNegativeDurationMinutes"),
    F.round(
        F.max("DurationMinutes"),
        2
    ).alias("ClosestToZeroDurationMinutes"),
    F.round(
        F.avg("DurationMinutes"),
        2
    ).alias("AverageNegativeDurationMinutes")
)

negative_duration_summary_df.show(
    truncate=False
)

+-----------------------+---------------------------+----------------------------+------------------------------+
|NegativeDurationRecords|MostNegativeDurationMinutes|ClosestToZeroDurationMinutes|AverageNegativeDurationMinutes|
+-----------------------+---------------------------+----------------------------+------------------------------+
|408                    |-493288.88                 |-0.12                       |-74094.87                     |
+-----------------------+---------------------------+----------------------------+------------------------------+



In [30]:
negative_duration_df.select(
    "deviceid",
    "bayid",
    "areaid",
    "arrival_time_raw",
    "departure_time_raw",
    "arrivaltime",
    "departuretime",
    "DurationMinutes"
).show(20, truncate=False)

+--------+-----+------+----------------------+----------------------+-------------------+-------------------+---------------+
|deviceid|bayid|areaid|arrival_time_raw      |departure_time_raw    |arrivaltime        |departuretime      |DurationMinutes|
+--------+-----+------+----------------------+----------------------+-------------------+-------------------+---------------+
|19020   |2532 |25    |02/23/2018 09:10:58 PM|02/23/2018 11:59:25 AM|2018-02-23 21:10:58|2018-02-23 11:59:25|-551.55        |
|17788   |1178 |19    |10/07/2018 02:59:21 AM|10/07/2018 03:00:55 AM|2018-10-07 03:59:21|2018-10-07 03:00:55|-58.43         |
|17789   |1179 |19    |10/07/2018 02:59:12 AM|10/07/2018 03:01:28 AM|2018-10-07 03:59:12|2018-10-07 03:01:28|-57.73         |
|27554   |5826 |24    |10/06/2019 02:59:59 AM|10/06/2019 03:50:59 AM|2019-10-06 03:59:59|2019-10-06 03:50:59|-9.0           |
|17244   |6069 |35    |10/07/2018 02:03:14 AM|10/07/2018 03:03:02 AM|2018-10-07 03:03:14|2018-10-07 03:03:02|-0.2     

In [31]:
negative_duration_df.withColumn(
    "DayDifference",
    F.datediff(
        F.col("departuretime"),
        F.col("arrivaltime")
    )
).groupBy(
    "DayDifference"
).count().orderBy(
    F.desc("count")
).show(20, truncate=False)

+-------------+-----+
|DayDifference|count|
+-------------+-----+
|0            |255  |
|-86          |29   |
|-189         |11   |
|-239         |7    |
|-139         |7    |
|-45          |6    |
|-1           |5    |
|-50          |5    |
|-190         |5    |
|-44          |4    |
|-41          |4    |
|-144         |4    |
|-87          |3    |
|-338         |3    |
|-143         |3    |
|-93          |3    |
|-38          |2    |
|-141         |2    |
|-91          |2    |
|-179         |2    |
+-------------+-----+
only showing top 20 rows


In [32]:
negative_duration_df.withColumn(
    "ArrivalDate",
    F.to_date("arrivaltime")
).groupBy(
    "ArrivalDate"
).count().orderBy(
    F.desc("count"),
    F.asc("ArrivalDate")
).show(20, truncate=False)

+-----------+-----+
|ArrivalDate|count|
+-----------+-----+
|2019-10-06 |51   |
|2018-03-19 |37   |
|2018-10-07 |26   |
|2018-12-04 |21   |
|2019-05-08 |13   |
|2018-07-06 |12   |
|2019-04-14 |8    |
|2019-07-26 |8    |
|2018-04-21 |7    |
|2019-06-08 |7    |
|2019-07-28 |7    |
|2018-06-29 |6    |
|2019-03-04 |6    |
|2018-06-26 |5    |
|2018-06-27 |5    |
|2018-07-03 |5    |
|2019-09-16 |5    |
|2019-05-02 |4    |
|2019-06-11 |4    |
|2019-10-19 |4    |
+-----------+-----+
only showing top 20 rows


In [33]:
negative_duration_df.unpersist()

DataFrame[deviceid: bigint, arrivaltime: timestamp, departuretime: timestamp, streetmarker: string, signplateid: string, sign: string, streetid: int, betweenstreet1id: int, betweenstreet2id: int, sideofstreet: int, sidename: string, bayid: int, inviolation: boolean, vehiclepresent: boolean, areaid: int, arrival_time_raw: string, departure_time_raw: string, device_id_raw: string, DurationMinutes: double, DepartureBeforeArrival: boolean]

6) Explain how records with missing or invalid departure information will be handled in subsequent duration-based queries.

In [34]:
parking_clean_df = parking_clean_df.withColumn(
    "HasValidDuration",
    F.when(
        F.col("arrivaltime").isNotNull()
        & F.col("departuretime").isNotNull()
        & (
            F.col("departuretime")
            >= F.col("arrivaltime")
        ),
        True
    ).otherwise(False)
)

In [35]:
parking_clean_df = parking_clean_df.withColumn(
    "ValidDurationMinutes",
    F.when(
        F.col("HasValidDuration"),
        F.col("DurationMinutes")
    ).otherwise(
        F.lit(None).cast("double")
    )
)

A total of 408 records had a departure timestamp earlier than the arrival
timestamp, representing approximately 0.00056% of all parking records. Of
these, 255 occurred on the same calendar day.

A noticeable concentration occurred on 7 October 2018 and 6 October 2019,
which correspond to the beginning of daylight saving time in Melbourne.
During timestamp parsing, some local times in the skipped 02:00–03:00 period
were shifted forward, producing apparent negative durations. Other records
contained date differences of several days or months, indicating likely
sensor-event matching or source-data errors.

These records were retained for data-quality reporting but excluded from all
duration-based queries because their true parking durations cannot be
determined reliably.

Records with missing or unparseable departure information produce a null
duration and are excluded from duration-based calculations. They may still be
retained in analyses that do not use parking duration, provided that they
satisfy the validity requirements of the corresponding query.

7) Display and print the schema for DeviceId, ArrivalTime, DepartureTime and DurationMinutes.

In [36]:
parking_validation_display_df = parking_clean_df.select(
    F.col("deviceid").alias("DeviceId"),
    F.col("arrivaltime").alias("ArrivalTime"),
    F.col("departuretime").alias("DepartureTime"),
    F.col("DurationMinutes")
)

In [37]:
parking_validation_display_df.show(
    10,
    truncate=False
)

+--------+-------------------+-------------------+---------------+
|DeviceId|ArrivalTime        |DepartureTime      |DurationMinutes|
+--------+-------------------+-------------------+---------------+
|17193   |2018-10-23 06:49:29|2018-10-23 06:54:24|4.92           |
|17193   |2018-02-07 19:30:00|2018-02-07 19:56:23|26.38          |
|17195   |2018-11-22 07:22:38|2018-11-22 07:28:22|5.73           |
|17195   |2018-09-18 21:24:56|2018-09-18 21:27:44|2.8            |
|17195   |2018-01-04 00:00:00|2018-01-04 03:45:46|225.77         |
|17186   |2018-07-13 00:00:00|2018-07-13 06:06:43|366.72         |
|17187   |2018-01-29 19:07:39|2018-01-29 19:11:53|4.23           |
|17195   |2018-10-20 06:29:15|2018-10-20 07:15:02|45.78          |
|17194   |2018-08-06 18:42:28|2018-08-06 18:49:11|6.72           |
|17195   |2018-01-03 07:02:17|2018-01-03 07:05:08|2.85           |
+--------+-------------------+-------------------+---------------+
only showing top 10 rows


In [38]:
parking_validation_display_df.printSchema()

root
 |-- DeviceId: long (nullable = true)
 |-- ArrivalTime: timestamp (nullable = true)
 |-- DepartureTime: timestamp (nullable = true)
 |-- DurationMinutes: double (nullable = true)



The displayed schema confirms that `DeviceId` was converted to a numeric
type, `ArrivalTime` and `DepartureTime` were parsed as timestamps, and
`DurationMinutes` was stored as a double value rounded to two decimal places.
The displayed sample records were used to verify that the transformations
were applied successfully.

8) Display a validation summary table containing the count for each of the following:
- total parking records;
- ArrivalTime parsing failures;
- DepartureTime parsing failures;
- DeviceId conversion failures;
- records with missing DepartureTime;
- records where DepartureTime is earlier than ArrivalTime.


In [39]:
validation_summary_df = parking_clean_df.agg(
    F.count("*").alias(
        "TotalParkingRecords"
    ),

    F.sum(
        F.when(
            F.col("arrival_time_raw").isNotNull()
            & F.col("arrivaltime").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "ArrivalTimeParsingFailures"
    ),

    F.sum(
        F.when(
            F.col("departure_time_raw").isNotNull()
            & F.col("departuretime").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "DepartureTimeParsingFailures"
    ),

    F.sum(
        F.when(
            F.col("device_id_raw").isNotNull()
            & F.col("deviceid").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "DeviceIdConversionFailures"
    ),

    F.sum(
        F.when(
            F.col("departure_time_raw").isNull(),
            1
        ).otherwise(0)
    ).alias(
        "MissingDepartureTimeRecords"
    ),

    F.sum(
        F.when(
            F.col("DepartureBeforeArrival"),
            1
        ).otherwise(0)
    ).alias(
        "DepartureBeforeArrivalRecords"
    )
)

In [40]:
validation_summary_df.show(
    truncate=False
)

+-------------------+--------------------------+----------------------------+--------------------------+---------------------------+-----------------------------+
|TotalParkingRecords|ArrivalTimeParsingFailures|DepartureTimeParsingFailures|DeviceIdConversionFailures|MissingDepartureTimeRecords|DepartureBeforeArrivalRecords|
+-------------------+--------------------------+----------------------------+--------------------------+---------------------------+-----------------------------+
|72912582           |0                         |0                           |0                         |0                          |408                          |
+-------------------+--------------------------+----------------------------+--------------------------+---------------------------+-----------------------------+



The validation summary was calculated using a single aggregation over the
cleaned parking DataFrame. Parsing and conversion failures were defined as
records where the original non-null value produced a null transformed value.
This distinguishes genuinely missing source values from invalid values that
failed during conversion.

No records were removed during data cleansing. Records with missing, invalid
or chronologically inconsistent timestamps were retained for data-quality
reporting but excluded from duration-based analyses where appropriate.

### Data Cleansing and Validation Summary

Missing-value profiling found that empty CSV fields had already been
interpreted as Spark null values in `signplateid`, `sign` and
`betweenstreet2id`. One blank `areaname` value was identified and converted
to null. No additional textual missing-value representations, such as `NA`,
`N/A`, `null`, `unknown`, `-` or `?`, were observed.

`ArrivalTime` and `DepartureTime` were successfully parsed using the format
`MM/dd/yyyy hh:mm:ss a`, and `DeviceId` was successfully converted to a
numeric type. No timestamp parsing failures, DeviceId conversion failures or
missing DepartureTime records were found.

A total of 408 records had a DepartureTime earlier than ArrivalTime,
representing approximately 0.00056% of all parking records. Some anomalies
were concentrated on the Melbourne daylight-saving transition dates
7 October 2018 and 6 October 2019. Other records contained much larger date
differences, indicating likely source-data or sensor-event matching errors.

No records were removed. Records with invalid durations were retained for
data-quality reporting and analyses that do not depend on duration. They were
marked as having an invalid duration, and their `ValidDurationMinutes` value
was set to null so that they are excluded from subsequent duration-based
queries.

### 1.3 Partition and Spark UI Evidence <a class="anchor" name="1.3"></a>
1.3.1) Display the total number of partitions within the cleaned parking DataFrame.

In [43]:
parking_partition_count = (
    parking_clean_df
    .rdd
    .getNumPartitions()
)

print(
    "Total number of partitions in "
    f"parking_clean_df: {parking_partition_count}"
)

Total number of partitions in parking_clean_df: 69


1.3.2) Print the number of records in each partition. Immediately before running this action, set the following Spark Job description: sc.setJobDescription(f"A1-P1-{student_auth_code}")

In [44]:
partition_count_rdd = (
    parking_clean_df
    .select("deviceid")
    .rdd
    .mapPartitionsWithIndex(
        lambda partition_id, rows: [
            (
                partition_id,
                sum(1 for _ in rows)
            )
        ]
    )
)

In [45]:
sc.setJobDescription(
    f"A1-P1-{student_auth_code}"
)

partition_record_counts = (
    partition_count_rdd.collect()
)

sc.setJobDescription(None)

In [46]:
partition_record_counts = sorted(
    partition_record_counts,
    key=lambda item: item[0]
)

for partition_id, record_count in partition_record_counts:
    print(
        f"Partition {partition_id}: "
        f"{record_count:,} records"
    )

Partition 0: 1,093,088 records
Partition 1: 1,045,808 records
Partition 2: 1,043,416 records
Partition 3: 1,054,580 records
Partition 4: 1,095,648 records
Partition 5: 1,064,542 records
Partition 6: 1,073,063 records
Partition 7: 1,085,498 records
Partition 8: 1,069,231 records
Partition 9: 1,043,017 records
Partition 10: 1,049,272 records
Partition 11: 1,059,697 records
Partition 12: 1,051,963 records
Partition 13: 1,052,721 records
Partition 14: 1,047,821 records
Partition 15: 1,069,659 records
Partition 16: 1,051,222 records
Partition 17: 1,069,043 records
Partition 18: 1,072,379 records
Partition 19: 1,042,332 records
Partition 20: 1,053,826 records
Partition 21: 1,072,522 records
Partition 22: 1,053,381 records
Partition 23: 1,059,832 records
Partition 24: 1,058,740 records
Partition 25: 1,061,174 records
Partition 26: 1,056,094 records
Partition 27: 1,060,892 records
Partition 28: 1,059,880 records
Partition 29: 1,069,792 records
Partition 30: 1,068,101 records
Partition 31: 1,06

In [47]:
import statistics

record_counts = [
    record_count
    for _, record_count in partition_record_counts
]

total_partition_records = sum(record_counts)
average_partition_records = (
    total_partition_records / len(record_counts)
)
minimum_partition_records = min(record_counts)
maximum_partition_records = max(record_counts)
partition_standard_deviation = statistics.pstdev(
    record_counts
)
partition_cv_percentage = (
    partition_standard_deviation
    / average_partition_records
    * 100
)

print(
    "Number of partitions:",
    len(record_counts)
)
print(
    "Total records across partitions:",
    f"{total_partition_records:,}"
)
print(
    "Average records per partition:",
    f"{average_partition_records:,.2f}"
)
print(
    "Minimum records in a partition:",
    f"{minimum_partition_records:,}"
)
print(
    "Maximum records in a partition:",
    f"{maximum_partition_records:,}"
)
print(
    "Partition record-count standard deviation:",
    f"{partition_standard_deviation:,.2f}"
)
print(
    "Coefficient of variation:",
    f"{partition_cv_percentage:.2f}%"
)

Number of partitions: 69
Total records across partitions: 72,912,582
Average records per partition: 1,056,704.09
Minimum records in a partition: 580,174
Maximum records in a partition: 1,105,466
Partition record-count standard deviation: 59,500.14
Coefficient of variation: 5.63%


1.3.3) Include a set of screenshots from the Spark UI tab that clearly show:  
- the job tab with the specified job description;
- the relevant stage and its number of tasks
- the task input size, input records or task-duration distribution.

<img src="screenshots/part1/p1_jobs.png" width="100%">
Figure 1: Spark Job with the required description

Figure 1 shows Job 41 with the required job description
`A1-P1-kzha0139`. The job completed one stage and all 69 tasks successfully.*


<img src="screenshots/part1/p1_stages.png" width="100%">
Figure 2: Relevant Spark stage

Figure 2 shows Stage 55, which processed 8.6 GiB of input using 69 tasks.
No shuffle read or shuffle write was required.


<img src="screenshots/part1/p1_stage55_details.png" width="100%">
Figure 3: Task input and duration distribution

Figure 3 shows the Stage 55 task timeline and summary metrics. The stage
processed 72,912,582 records, and most tasks received approximately
128.1 MiB of input and completed in approximately 5 seconds.

1.3.4) Analyse the figures in the screenshot and briefly comment on whether the workload appears evenly distributed across the partitions. If not, why?

The partition workload appears broadly balanced. Stage 55 processed
72,912,582 records (8.6 GiB) across 69 tasks. The 25th percentile, median and
75th percentile input sizes were all approximately 128.1 MiB, while their
record counts were 1,054,410, 1,060,997 and 1,070,071 respectively. This
shows that most tasks received a similar amount of data.

Task durations were also consistent: the median and both middle quartiles
were approximately 5 seconds, with an overall range of 3–6 seconds. There
was no long-running task suggesting significant data skew.

The minimum partition contained 69.1 MiB and 580,174 records. This was likely
the final residual file split, which was smaller because the input file size
was not an exact multiple of the target partition size. Therefore, the
workload was sufficiently balanced despite one naturally smaller final
partition.

1.3.5) Provide a concise explanation detailing how Spark determines the initial number and approximate size of input partitions when reading from a CSV file.

Spark determines the initial CSV input partitions primarily from the input
file size and the configured maximum number of bytes per file partition.
For file-based data sources, the relevant setting is
`spark.sql.files.maxPartitionBytes`, whose default value is approximately
128 MiB. Spark may also consider the file open cost, filesystem block
information and the configured minimum or maximum partition numbers.

Because CSV is a splittable, line-oriented format, Spark divided the 8.6 GiB
parking file into approximately 128 MiB input splits while preserving record
boundaries. This produced 69 input partitions. Most partitions were about
128.1 MiB, while the final partition was only 69.1 MiB because it contained
the remaining bytes at the end of the file.

The `local[4]` setting does not determine the number of input partitions. It
controls how many tasks can execute concurrently. The narrow cleansing
transformations did not require a shuffle or change the original input
partition count, so the cleaned parking DataFrame retained the 69 CSV input
partitions.

## Part 2. Working with DataFrames (35%) <a class="anchor" name="2-dataframes"></a>
In this section, you need to use DataFrame functions to answer the queries.

2.1 Create the time attributes  
2.1.1) From ArrivalTime, create columns for: year; month; day of week;  arrival hour and arrival minute;.

2.1.2) With day of week and hour and minutes, create a new column named isPeak and set the value to True or False (i.e. True = Peak Hour, False = Non-peak hour).  
- Peak hours are:
- Weekdays 7:00 - 9:00 am (excluding 9:00 am) and 4:00 - 6:30 pm ( excluding 6:30 pm).
- Non-peak hours are times outside these periods.
- Represent day of week using Spark’s convention: 1 = Sunday, 2 = Monday……

2.1.3) Display five rows containing ArrivalTime, year, month, day of week, arrival hour, arrival minute and isPeak, and print the relevant schema.

2.2 Join the cleaned parking sensor data with both lookup tables to add the area and street names. (Note:The area and street lookup table is provided in JSON format).  
Select an appropriate join type;
- Report the number of parking records that do not match an area look up row and the number that do not match a street lookup row. 
- Identify a suitable join strategy for these joins and justify your choice.
- Display the first five joined records with the following columns: DeviceId, AreaId, AreaName, StreetId, and StreetName.

### 2.3 Compare Parking Activity and Violation Rates
The council would like to compare parking activity and violation patterns during peak and off-peak periods.  
For each parking area:
- PeakDailyAverage: the average number of valid peak-hour sessions per active weekday.
- PeakViolationRate: percentage of valid peak-hour sessions marked as InViolation.
- OffPeakDailyAverage: the average number of valid off-peak sessions per active day.
- OffPeakViolationRate: the percentage of valid off-peak sessions marked as InViolation.
- TotalValidSessions: total peak and off-peak sessions.
1) Display the ten areas with the highest TotalValidSessions, ordered by TotalValidSessions descending. If counts are equal, order by AreaId ascending. 
1) Briefly compare the peak and off-peak results and discuss whether higher parking activity appears to correspond with higher violation rates among these areas.
Note:  
- Violation rate = (sessions where Inviolation is true) / (sessions with a valid Inviolation value in the same period) * 100%.
- A valid session must have a non-null AreaID, and a successfully parsed ArrivalTime. 
- Calculate each violation rate using sessions with a valid InViolation value in the corresponding period.
- A missing DepartureTime does not invalidate a session because duration is not used in this task.
- An active day is a date on which the area has at least one valid parking session.


In [41]:
#2.3.1

In [42]:
#2.3.2

### 2.4 Identify high-use and long-stay parking bays  
Parking planners would like to identify bays with frequent vehicle turnover and bays that are commonly occupied for longer periods.  
For each BayId, calculate:  
1. the total number of valid parking sessions;
1. the total number of sessions with a valid duration;
1. the average parking duration in minutes, rounded to two decimal places
1. the percentage of sessions marked as InViolation, rounded to two decimal places  
Identify and display:  
1. the top 5 bays with the highest turnover, order the turnover result by totalSessions descending, use BayID ascending to resolve ties.
1. the top 5 bays with the longest average stay, order the longest-stay result by AverageDurationMinutes descending, use BayID ascending to resolve ties.
1. For both top-5 results, display BayID, TotalSessions, ValidDurationSessions, AverageDurationMinutes and ViolationRates.
Note:  
- A valid session must have a non-null identifier required by the query, a successfully parsed ArrivalTime.
- Define turnover as the total number of valid sessions recorded for a bay during the observation period


### 2.5 Combine parking activity with traffic volume
The council would like to identify when an area experiences unusually high traffic and parking activity at the same time.  
The traffic count dataset is measured by sensors during 10-minute time blocks. A time block begins at 00, 10, 20, 30, 40, or 50 minutes past the hour. For example, times from 00:00:00 inclusive to 00:10:00 exclusive belong to the block beginning at 00:00:00. A traffic record with: 2018, 1, 1, 0, 0 is the first 10-minute block on 1/1/2018, and belongs to the block 2018-01-01 00:00:00 to 2018-01-01 00:09:59.  
For each area-time block:  
- Define ParkingSessions as the number of parking sessions beginning within the block.
- Define TrafficVolume as the sum of traffic_count within the block.

1. Now, you need to think of a strategy to combine parking activity and traffic volume. Select an appropriate join type and justify your choice. 

2. Across the matched area-time blocks, define high parking activity and high traffic volume as values greater than their respective averages.  
Display a summary containing:  
the average ParkingSessions, average TrafficVolume, number of matched area-time blocks, and the number where both measures are above average.   


3. Display the ten above-average combinations with the highest ParkingSessions, display AreaID, AreaName, BlockStart, ParkingSessions, TrafficVolume, order the result by ParkingSessions and TrafficVolume in descending order. If values are equal, order by AreaId and BlockStart in ascending order.

4. Before running the query, you must set:  
sc.setJobDescription(f"A1-P2-TrafficJoin-{student_auth_code}")  
Students must include:  
- the formatted physical plan using explain(mode="formatted");
- Spark UI screenshots showing the job description, any Exchange, and the relevant shuffle-read, shuffle-write values.
- a short explanation of why data movement was or was not required.


### 2.6 Exploratory analysis  <a class="anchor" name="2.6"></a>
Produce two suitable plots exploring relationships among parking activity, duration, violation rate, or traffic volume.  
For each plot:
1. State the question being investigated;
1. Display the first five rows and the row count of the aggregated dataset used to create the plot
1. Select the appropriate plot type and briefly justify your choice.
1. Explain the observed pattern shown by the plot.
1. identify one limitation of the interpretation.


Write your dicsussion here.

## Part 3 Query Optimisation (25%) <a class="anchor" name="part-3"></a>
Assume that the following monthly parking report will be generated regularly. A correct query is required first, but it should also execute efficiently on the full dataset. In this section, implement the report, inspect how Spark executes it, and evaluate the optimization.


Monthly parking report  
For each month, identify the five areas with the highest number of peak-hour parking sessions among areas whose peak-hour violation rate is greater than the monthly violation benchmark.  
Area Violation Rate = (Peak-hour sessions where inViolation is true) / (Peak-hour sessions with a valid inViolation value) * 100   
Monthly Violation Benchmark = (Peak-hour sessions where InViolation is true across all areas in the month) / (Peak-hour sessions with a valid InViolation value across all areas in the same month) * 100  
  
Display:  
- month;
- area id;
- area name;
- park session count;
- violation count;
- violation rate;
- monthly rank.  
Report the number of months and total result rows, Display the complete result ordered by Month and Monthly Rank in ascending order.

##### 3.1 DataFrame API and Spark SQL 
Implement the monthly report independently using:  
1. the PySpark DataFrame API;
2. Spark SQL  
Both implementations must begin with the same prepared input data. Do not create the SQL result by registering the completed DataFrame API result as a temporary view.  
  
Complete the following:  
1. Report the row count returned by each implementation
2. Confirm that the results are identical using exceptAll() in both directions, and display both mismatch counts.
3. For each implementation, display the first 20 result rows using the following column order: 
- Month
- AreaId
- AreaName
- PeakSessionCount
- ViolationCount
- ViolationRate
- MonthlyViolationBenchmark
- MonthlyRank
4. Order the displayed results by Month and MonthlyRank in ascending order.
5. Inspect the physical plan for each implementation, and briefly compare the operations. Briefly explain whether Spark selected different execution strategies, and why?


##### 3.2 Baseline plan and Spark UI analysis  
Select either SQL or DataFrame API from Section 3.1 as a baseline. Clearly state which implementation you selected.  
Before running the baseline query, set: sc.setJobDescription(f"A1-P3-2-Baseline-{student_auth_code}")  
Execute an action that materialises the complete final result and record the wall-clock execution time. Use the same action later when evaluating the optimised query.  
Submit and include:  
- the formatted physical plan, you may refer to the plan already printed in Section 3.1 rather than printing it again.
- Spark UI screenshots clearly showing: the job description, the SQL/DAG, the relevant stage details, relevant shuffle-read, shuffle-write figures, where applicable.
- the measured execution time;  
In no more than 300 words:  
1. Identify the most expensive stages or operations.
2. Support the explanation using numbers visible in your screenshots.
3. Suggest an improvement strategy.


Write your discussion here.

##### 3.3 Evaluate one optimisation  
Implement one optimisation based on the evidence and reasoning presented in Section 3.2. Briefly state what you changed and why you expect it to improve execution.  
Before executing it, set: sc.setJobDescription(f"A1-P3-3-Optimised-{student_auth_code}")  
Then:  
1. Run the baseline and revised queries under the same conditions;
2. Confirm that the optimised query produces exactly the same result as the baseline
3. Record the execution time and collect the corresponding Spark UI evidence.
4. Decide whether the optimisation should be retained, justify your decision.
5. A technically reasonable improvement and evidence-based evaluation are required for full marks.
